In [6]:
import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from dotenv import load_dotenv

In [4]:
!uv add matplotlib

Resolved 188 packages in 601ms                                       
Prepared 6 packages in 1.72s                                             
Installed 6 packages in 25ms                                
 + contourpy==1.3.2
 + cycler==0.12.1
 + fonttools==4.62.1
 + kiwisolver==1.5.0
 + matplotlib==3.10.8
 + pyparsing==3.3.2


In [7]:
load_dotenv()

True

In [9]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# 이미지, 영상, 텍스트, 음성에 대한 고차원 데이터 -> vector space
# 임베딩은 고차원 데이터의 저차원 압축 표현

In [11]:
llm = ChatOpenAI(model='gpt-4o-mini')
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [12]:
test_emb = embeddings_model.embed_query('Hello')

In [14]:
len(test_emb)

1536

In [15]:
!uv add tiktoken

Resolved 188 packages in 163ms                                       
Audited 177 packages in 44ms                                         


In [16]:
import tiktoken

In [23]:
# enc 인코더
enc = tiktoken.encoding_for_model('gpt-4o-mini') # embedding space, representation sapce

In [18]:
text = "안녕하세요, 오늘 LLM에 대해 배워보겠습니다"
tokens = enc.encode(text)

In [19]:
tokens

[14307,
 171731,
 11,
 106820,
 451,
 19641,
 3107,
 67946,
 33628,
 33771,
 8122,
 105216]

In [20]:
enc.decode(tokens)

'안녕하세요, 오늘 LLM에 대해 배워보겠습니다'

In [ ]:
# text => 숫자로 변환 tokenize => 그 다음 백터 처리

In [22]:
for i, token_id in enumerate(tokens):
    token_text = enc.decode([token_id])
    print(f"token{i+1} : ID : {token_id} -> {token_text}")

token1 : ID : 14307 -> 안
token2 : ID : 171731 -> 녕하세요
token3 : ID : 11 -> ,
token4 : ID : 106820 ->  오늘
token5 : ID : 451 ->  L
token6 : ID : 19641 -> LM
token7 : ID : 3107 -> 에
token8 : ID : 67946 ->  대해
token9 : ID : 33628 ->  배
token10 : ID : 33771 -> 워
token11 : ID : 8122 -> 보
token12 : ID : 105216 -> 겠습니다


In [ ]:
# LLM tokenizing 방식 처리
# ex) I like go to school : I, like, go, to, school 스페이스 기준으로 예전에 진행함

# BPE : Byte Pair Encoding

In [24]:
long_text = """인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를 보여주고 있습니다.
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다."""

In [ ]:
# chunk
# Splitter 문서들을 chunck 단위로 나눠주는 class
# CharacterTextSplitter 캐릭터 단위로 나눠줌
# RecursiveCharacterTextSplitter
# from_tiktoken_encoder()

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [27]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20, separators=["\n\n", "\n", ". ", "", " "]) #separators 우선 순위를 주는 것

In [28]:
chunks = splitter.split_text(long_text)

In [29]:
len(chunks)

7

In [32]:
for i, chunk in enumerate(chunks):
    print(f"[chunk {i+1}] : {len(chunk)} characters")
    print(chunk)

[chunk 1] : 65 characters
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
[chunk 2] : 46 characters
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다
[chunk 3] : 65 characters
. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를 보여주고 있습니다.
[chunk 4] : 46 characters
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다
[chunk 5] : 94 characters
. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
[chunk 6] : 40 characters
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다
[chunk 7] : 61 characters
. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다.


In [35]:
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10, separators=["\n\n", "\n", ". ", "", " "]) #separators 우선 순위를 주는 것

In [36]:
chunks = splitter.split_text(long_text)

In [37]:
for i, chunk in enumerate(chunks):
    print(f"[chunk {i+1}] : {len(chunk)} characters")
    print(chunk)

[chunk 1] : 50 characters
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적
[chunk 2] : 24 characters
지각능력을 인공적으로 구현하려는 기술입니다.
[chunk 3] : 46 characters
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다
[chunk 4] : 50 characters
. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인
[chunk 5] : 24 characters
분야에서 혁신적인 성과를 보여주고 있습니다.
[chunk 6] : 46 characters
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다
[chunk 7] : 50 characters
. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을 제공하며, RAG(Retri
[chunk 8] : 49 characters
RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용
[chunk 9] : 13 characters
구축에 특히 유용합니다.
[chunk 10] : 40 characters
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다
[chunk 11] : 49 characters
. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를 검색하여
[chunk 12] : 20 characters
청크를 검색하여 LLM에 전달합니다.


In [39]:
# chunk_size, chunk_overlap 값들에 따라서 chunk처리가 되는 것
splitter_tiktoken = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name = 'gpt-4o-mini', chunk_size=50, chunk_overlap=10
)

In [40]:
chunks = splitter_tiktoken.split_text(long_text)
for i, chunk in enumerate(chunks):
    print(f"[chunk {i+1}] : {len(chunk)} characters")
    print(chunk)

[chunk 1] : 65 characters
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
[chunk 2] : 100 characters
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를
[chunk 3] : 27 characters
처리 분야에서 혁신적인 성과를 보여주고 있습니다.
[chunk 4] : 80 characters
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을
[chunk 5] : 72 characters
벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
[chunk 6] : 84 characters
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를
[chunk 7] : 31 characters
후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다.


In [41]:
def count_token(text, model='gpt-4o-mini'):
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

In [42]:
chunks = splitter_tiktoken.split_text(long_text)
for i, chunk in enumerate(chunks):
    tokens = count_token(chunk)
    print(f"[chunk {i+1}] : {tokens} tokens")
    print(chunk)

[chunk 1] : 40 tokens
인공지능(AI)은 컴퓨터 과학의 한 분야로, 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 기술입니다.
[chunk 2] : 47 tokens
최근 대규모 언어 모델(LLM)의 발전으로 AI 기술은 급격한 변화를 겪고 있습니다. GPT, Claude, Gemini 등의 모델이 등장하며 자연어 처리 분야에서 혁신적인 성과를
[chunk 3] : 15 tokens
처리 분야에서 혁신적인 성과를 보여주고 있습니다.
[chunk 4] : 47 tokens
LangChain은 이러한 LLM을 활용한 애플리케이션 개발을 돕는 프레임워크입니다. 문서 로딩, 텍스트 분할, 임베딩, 벡터 검색 등의 기능을
[chunk 5] : 27 tokens
벡터 검색 등의 기능을 제공하며, RAG(Retrieval-Augmented Generation) 시스템 구축에 특히 유용합니다.
[chunk 6] : 49 tokens
RAG는 외부 지식을 LLM에 제공하여 답변의 정확성을 높이는 기술입니다. 문서를 청크로 나누고, 벡터 데이터베이스에 저장한 후, 질문과 관련된 청크를
[chunk 7] : 17 tokens
후, 질문과 관련된 청크를 검색하여 LLM에 전달합니다.


In [43]:
test_text = """서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한 대중교통 시스템을 갖추고 있습니다. 지하철 9개 노선과 수천 대의 버스가 시민들의 이동을 돕고 있습니다."""

In [45]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 테스트 케이스 3개만
configs = [
    (50, 10),
    (100, 20),
    (200, 50),
]

for chunk_size, overlap in configs:
    print("\n" + "="*40)
    print(f"chunk_size={chunk_size}, overlap={overlap}")
    print("="*40)

    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        model_name="gpt-4o-mini",
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )

    chunks = splitter.split_text(test_text)

    print(f"총 chunk 개수: {len(chunks)}\n")

    for i, chunk in enumerate(chunks):
        print(f"[chunk {i+1}] ({count_token(chunk)} tokens)")
        print(chunk)
        print("-"*20)


chunk_size=50, overlap=10
총 chunk 개수: 4

[chunk 1] (49 tokens)
서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁,
--------------------
[chunk 2] (50 tokens)
있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 랜드마크가 공존합니다. 교통 면에서
--------------------
[chunk 3] (49 tokens)
공존합니다. 교통 면에서 서울은 세계적으로 우수한 대중교통 시스템을 갖추고 있습니다. 지하철 9개 노선과 수천 대의 버스가 시민들의 이동을 돕고
--------------------
[chunk 4] (12 tokens)
버스가 시민들의 이동을 돕고 있습니다.
--------------------

chunk_size=100, overlap=20
총 chunk 개수: 2

[chunk 1] (97 tokens)
서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한
--------------------
[chunk 2] (55 tokens)
랜드마크가 공존합니다. 교통 면에서 서울은 세계적으로 우수한 대중교통 시스템을 갖추고 있습니다. 지하철 9개 노선과 수천 대의 버스가 시민들의 이동을 돕고 있습니다.
--------------------

chunk_size=200, overlap=50
총 chunk 개수: 1

[chunk 1] (133 tokens)
서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁, 창덕궁

In [49]:
def split_and_report(text, chunk_size=50, chunk_overlap=10):
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        model_name="gpt-4o-mini",
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )

    chunks = splitter.split_text(text)
    token_counts = [count_token(c) for c in chunks]

    print(f"chunking report (chunk_size = {chunk_size}, chunk_overlap = {chunk_overlap})")
    # zip은 동일한 
    for i, (chunk, tc) in enumerate(zip(chunks, token_counts)):
        first_line = chunk.split('\n')[0][:40]
        print(f"[{i+1}] {tc} tokens, {len(chunk)} characters | {first_line}")

    print(f"summary : {len(chunks)} chunks")
    print(f"avg : {np.mean(token_counts)}")
    print(f"max : {max(token_counts)}")

In [48]:
split_and_report(test_text)

chunking report (chunk_size = 50, chunk_overlap = 10
[1] 49 tokens, 83 characters | 서울은 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북
[2] 51 tokens, 83 characters | 대한민국의 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나
[3] 50 tokens, 79 characters | 수도이자 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 
[4] 51 tokens, 79 characters | 최대 도시입니다. 한강을 중심으로 강남과 강북으로 나뉘며, 약 1000만
[5] 49 tokens, 69 characters | 중심으로 강남과 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있
[6] 48 tokens, 67 characters | 강북으로 나뉘며, 약 1000만 명의 인구가 거주하고 있습니다. 서울에는
[7] 49 tokens, 70 characters | 약 1000만 명의 인구가 거주하고 있습니다. 서울에는 경복궁, 창덕궁 
[8] 48 tokens, 68 characters | 명의 인구가 거주하고 있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁
[9] 50 tokens, 71 characters | 거주하고 있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타
[10] 50 tokens, 70 characters | 있습니다. 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데
[11] 49 tokens, 68 characters | 서울에는 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등
[12] 49 tokens, 69 characters | 경복궁, 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 
[13] 48 tokens, 68 characters | 창덕궁 등 조선시대 궁궐과 N서울타워, 롯데월드타워 등 현대적 랜드마크가
[14] 49

In [50]:
query = "인공지능이 세상을 바꾸고 있습니다"
query_vector = embeddings_model.embed_query(query)

In [51]:
query_vector

[0.0316162109375,
 0.028289794921875,
 0.0098724365234375,
 0.033111572265625,
 -0.0077667236328125,
 -0.042755126953125,
 -0.022674560546875,
 0.0775146484375,
 -0.0364990234375,
 -0.0211334228515625,
 -0.050750732421875,
 -0.00762939453125,
 0.0005621910095214844,
 -0.0809326171875,
 -0.024932861328125,
 -0.0245819091796875,
 -0.0897216796875,
 0.0189971923828125,
 0.047576904296875,
 -0.0170440673828125,
 -0.021453857421875,
 0.003841400146484375,
 0.008392333984375,
 -0.013671875,
 0.0179443359375,
 -0.00299072265625,
 0.0037517547607421875,
 0.0264434814453125,
 0.017852783203125,
 -0.0164642333984375,
 0.03216552734375,
 -0.0288848876953125,
 -0.0435791015625,
 -0.04534912109375,
 0.0268707275390625,
 0.019012451171875,
 0.00904083251953125,
 0.005352020263671875,
 -0.036529541015625,
 0.0155487060546875,
 0.0021572113037109375,
 0.01678466796875,
 0.0244903564453125,
 0.034912109375,
 0.0012111663818359375,
 0.0282135009765625,
 -0.06268310546875,
 0.000408172607421875,
 0.05676

In [ ]:
# 임베딩의 차원은 model이 결정함
# 임베딩의 길이가 클수록 좋음 (크면 의미를 풍부하게 담을 수 있기 때문에)

In [52]:
texts = [
    "AI가 세강을 바꾸고 있습니다",
    "AI가 의료분야를 혁신하고 있습니다",
    "머신러닝으로 질변을 예측할 수 있습니다",
    "오늘 저녁에 치킨을 시켜먹었다"
]

doc_vectors = embeddings_model.embed_documents(texts)

In [53]:
doc_vectors

[[0.0013360977172851562,
  0.035125732421875,
  0.0301055908203125,
  0.01922607421875,
  -0.0171356201171875,
  0.0035400390625,
  -0.033111572265625,
  0.01389312744140625,
  -0.007213592529296875,
  -0.0201263427734375,
  0.00978851318359375,
  -0.0134735107421875,
  0.0250396728515625,
  -0.00913238525390625,
  -0.0299072265625,
  -0.054656982421875,
  0.0011081695556640625,
  0.0223846435546875,
  -0.032379150390625,
  0.01384735107421875,
  -0.03851318359375,
  -0.011962890625,
  -0.043701171875,
  0.02508544921875,
  -0.03082275390625,
  -0.0311126708984375,
  0.059814453125,
  0.0010089874267578125,
  -0.004169464111328125,
  -0.006092071533203125,
  0.009552001953125,
  -0.030731201171875,
  -0.0212249755859375,
  -0.047637939453125,
  0.0027675628662109375,
  0.0255126953125,
  0.00316619873046875,
  0.00951385498046875,
  -0.03271484375,
  0.01181793212890625,
  0.029510498046875,
  -0.0090484619140625,
  -0.003986358642578125,
  0.018096923828125,
  0.052825927734375,
  0.0

In [54]:
len(doc_vectors)

4

In [55]:
for vec in doc_vectors:
    print(len(vec))

1536
1536
1536
1536


In [ ]:
# 임베딩으로 변경한 이유는 계산을 하기 위해서 진행한 것

In [59]:
from sklearn.metrics.pairwise import cosine_similarity

In [57]:
!uv add scikit-learn

Resolved 194 packages in 485ms                                       
Prepared 4 packages in 4.67s                                             
Installed 4 packages in 65ms                                
 + joblib==1.5.3
 + scikit-learn==1.7.2
 + scipy==1.15.3
 + threadpoolctl==3.6.0


In [60]:
cosine_similarity(doc_vectors)

array([[1.        , 0.47284327, 0.25677045, 0.13213351],
       [0.47284327, 1.        , 0.25124074, 0.1494312 ],
       [0.25677045, 0.25124074, 1.        , 0.09617178],
       [0.13213351, 0.1494312 , 0.09617178, 1.        ]])

In [61]:
all_vectors = [query_vector] + doc_vectors
all_texts = [query] + texts

cosine_similarity(all_vectors)

array([[1.        , 0.46772993, 0.32556304, 0.19505497, 0.05781237],
       [0.46772993, 1.        , 0.47284327, 0.25677045, 0.13213351],
       [0.32556304, 0.47284327, 1.        , 0.25124074, 0.1494312 ],
       [0.19505497, 0.25677045, 0.25124074, 1.        , 0.09617178],
       [0.05781237, 0.13213351, 0.1494312 , 0.09617178, 1.        ]])

In [63]:
similarity_matrix = cosine_similarity(all_vectors)

labels = ["query"] + [f"doc{i+1}" for i in range(len(texts))]

df = pd.DataFrame(
    similarity_matrix,
    index=labels,
    columns=labels
)

print(df)

          query      doc1      doc2      doc3      doc4
query  1.000000  0.467730  0.325563  0.195055  0.057812
doc1   0.467730  1.000000  0.472843  0.256770  0.132134
doc2   0.325563  0.472843  1.000000  0.251241  0.149431
doc3   0.195055  0.256770  0.251241  1.000000  0.096172
doc4   0.057812  0.132134  0.149431  0.096172  1.000000


In [68]:
categories = {
    "기술": ["인공지능과 머신러닝이 산업을 혁신하고 있다", "클라우드 서비스가 기업의 디지털 전환을 가속화한다"],
    "스포츠": ["프로야구 시즌이 시작되어 팬들이 열광하고 있다", "올림픽에서 한국 선수가 금메달을 획득했다"],
    "음식": ["이 레스토랑의 파스타가 정말 맛있었다", "한국의 김치는 세계적으로 유명한 발효 식품이다"],
}

def classify(text, categories):
    text_vec = embeddings_model.embed_query(text)

    scores = {}

    for category, sentences in categories.items():
        merged = " ".join(sentences)

        cat_vec = embeddings_model.embed_query(merged)

        sim = cosine_similarity([text_vec], [cat_vec])[0][0]
        scores[category] = sim

    best = max(scores, key=scores.get)

    if scores[best] < 0.3:
        return "기타", scores

    return best, scores

In [69]:
test_texts = [
    "AI 기술이 빠르게 발전하고 있다",
    "어제 야구 경기 진짜 재밌었다",
    "오늘은 피자랑 치킨 먹고 싶다",
]

for text in test_texts:
    result, scores = classify(text, categories)
    print("\n" + "="*30)
    print(f"입력: {text}")
    print(f"예측: {result}")
    print(f"점수: {scores}")


입력: AI 기술이 빠르게 발전하고 있다
예측: 기술
점수: {'기술': np.float64(0.36785439471431536), '스포츠': np.float64(0.28742234200312905), '음식': np.float64(0.1736543873895091)}

입력: 어제 야구 경기 진짜 재밌었다
예측: 스포츠
점수: {'기술': np.float64(0.06245234460349115), '스포츠': np.float64(0.4007906604312895), '음식': np.float64(0.153525021359655)}

입력: 오늘은 피자랑 치킨 먹고 싶다
예측: 음식
점수: {'기술': np.float64(0.06259019729514147), '스포츠': np.float64(0.13295557410647163), '음식': np.float64(0.3206403752864644)}


In [70]:
import numpy as np

categories = {
    "기술": [
        "인공지능과 머신러닝이 산업을 혁신하고 있다",
        "클라우드 서비스가 기업의 디지털 전환을 가속화한다",
    ],
    "스포츠": [
        "프로야구 시즌이 시작되어 팬들이 열광하고 있다",
        "올림픽에서 한국 선수가 금메달을 획득했다",
    ],
    "음식": [
        "이 레스토랑의 파스타가 정말 맛있었다",
        "한국의 김치는 세계적으로 유명한 발효 식품이다",
    ],
}

flattened_categories = [sent for sents in categories.values() for sent in sents]
categories_vectors = embeddings_model.embed_documents(flattened_categories)

def classify(text, categories):
    text_vector = embeddings_model.embed_query(text)
    similarities = cosine_similarity([text_vector], categories_vectors)[0]
    sents_index = np.argmax(similarities)
    category_index = sents_index // len(categories)
    return list(categories.keys())[category_index], flattened_categories[sents_index]


classify("이번 월드컵은 아르헨티나가 우승할것이다", categories)

('스포츠', '올림픽에서 한국 선수가 금메달을 획득했다')

In [81]:
categories = {
    "기술": ["인공지능과 머신러닝이 산업을 혁신하고 있다", "클라우드 서비스가 기업의 디지털 전환을 가속화한다"],
    "스포츠": ["프로야구 시즌이 시작되어 팬들이 열광하고 있다", "올림픽에서 한국 선수가 금메달을 획득했다"],
    "음식": ["이 레스토랑의 파스타가 정말 맛있었다", "한국의 김치는 세계적으로 유명한 발효 식품이다"],
}

def classify3(text, categories):
    cat_vectors = {}
    for cat, examples in categories.items():
        embs = embeddings_model.embed_documents(examples)
        # 평균처리
        cat_vectors[cat] = np.mean(embs, axis=0)

    text_emb = embeddings_model.embed_query(text)

    scores = {}
    for cat, cat_vec in cat_vectors.items():
        sim = cosine_similarity([text_emb], [cat_vec])[0][0]
        scores[cat] = round(sim, 4)

    best_cat = max(scores, key=scores.get)

    print(f"입력 : {text}")
    print(f"예측 : {best_cat}")
    for c, s in scores.items():
        marker = " <<<" if c == best_cat else ""
        print(f"{c} : {s}{marker}")

    return best_cat, scores

In [82]:
test_sentences = [
    "AI 기술이 빠르게 발전하고 있다",
    "어제 야구 경기 진짜 재밌었다",
    "오늘은 피자랑 치킨 먹고 싶다",
]

for sent in test_sentences:
    classify3(sent, categories)

입력 : AI 기술이 빠르게 발전하고 있다
예측 : 기술
기술 : 0.4936 <<<
스포츠 : 0.3259
음식 : 0.235
입력 : 어제 야구 경기 진짜 재밌었다
예측 : 스포츠
기술 : 0.1127
스포츠 : 0.4643 <<<
음식 : 0.2253
입력 : 오늘은 피자랑 치킨 먹고 싶다
예측 : 음식
기술 : 0.104
스포츠 : 0.1847
음식 : 0.3545 <<<


In [ ]:
# CachedBackedEmbeddings

In [89]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_core.stores import InMemoryByteStore

In [86]:
!uv add langchain_classic langchain_core

Resolved 194 packages in 429ms                                       
Audited 181 packages in 55ms                                         


In [92]:
store = InMemoryByteStore()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings_model, store, namespace="embedding-cache"
)